# 🤝 Personalized Networking Assistant — Colab Notebook

This notebook builds and runs the full project end-to-end inside Google Colab:

1. Install dependencies
2. Write out the project files (backend, frontend, tests)
3. Run the fast test suite
4. Launch the FastAPI backend
5. Launch the Streamlit frontend and expose it publicly via ngrok

Run every cell **top to bottom**. You'll need a free ngrok authtoken:
https://dashboard.ngrok.com/get-started/your-authtoken


## 1. Install dependencies

In [1]:
%%capture
!pip uninstall -y torch torchvision -q
!pip install -q fastapi==0.115.0 "uvicorn[standard]"==0.30.6 streamlit==1.38.0 \
    transformers==4.44.2 torch==2.4.0 torchvision==0.19.0 wikipedia==1.4.0 pydantic==2.9.2 \
    pytest==8.3.2 httpx==0.27.2 requests==2.32.3 pyngrok==7.2.0 nest-asyncio==1.6.0

## 2. Create the project structure

In [3]:
import os

PROJECT_ROOT = "/content/personalized-networking-assistant"
dirs = [
    PROJECT_ROOT,
    f"{PROJECT_ROOT}/backend",
    f"{PROJECT_ROOT}/backend/services",
    f"{PROJECT_ROOT}/frontend",
    f"{PROJECT_ROOT}/tests",
    f"{PROJECT_ROOT}/data",
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())


Working directory: /content/personalized-networking-assistant


## 3. Write out project files

In [4]:
%%writefile backend/__init__.py


Writing backend/__init__.py


In [5]:
%%writefile backend/models.py
"""Pydantic schemas used by the FastAPI backend."""
from typing import List, Optional
from pydantic import BaseModel, Field


class GenerateStartersRequest(BaseModel):
    event_description: str = Field(..., min_length=3, description="Description of the event/topic")
    interests: List[str] = Field(default_factory=list, description="User's stated interests")
    num_starters: int = Field(3, ge=1, le=5, description="How many conversation starters to return")


class StarterItem(BaseModel):
    id: int
    starter: str


class GenerateStartersResponse(BaseModel):
    themes: List[str]
    starters: List[StarterItem]


class FactCheckRequest(BaseModel):
    query: str = Field(..., min_length=2, description="Topic or claim to verify")


class FactCheckResponse(BaseModel):
    query: str
    found: bool
    summary: Optional[str] = None
    url: Optional[str] = None
    options: List[str] = Field(default_factory=list)


class FeedbackRequest(BaseModel):
    history_id: int
    useful: bool


class HistoryItem(BaseModel):
    id: int
    event_description: str
    interests: str
    themes: str
    starter: str
    useful: Optional[bool] = None
    created_at: str


Writing backend/models.py


In [6]:
%%writefile backend/database.py
"""Lightweight SQLite persistence for conversation history and feedback.

No ORM is used on purpose - the schema is tiny and this keeps the project
dependency-light and easy to run inside Google Colab.
"""
import sqlite3
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from typing import List, Optional, Sequence

DB_PATH = Path(__file__).resolve().parent.parent / "data" / "app.db"


def init_db(db_path: Optional[Path] = None) -> Path:
    """Create the database file and table if they do not already exist."""
    path = Path(db_path) if db_path else DB_PATH
    path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(path)
    try:
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS history (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                event_description TEXT NOT NULL,
                interests TEXT,
                themes TEXT,
                starter TEXT NOT NULL,
                useful INTEGER,
                created_at TEXT NOT NULL
            )
            """
        )
        conn.commit()
    finally:
        conn.close()
    return path


@contextmanager
def get_connection(db_path: Optional[Path] = None):
    path = Path(db_path) if db_path else DB_PATH
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row
    try:
        yield conn
    finally:
        conn.close()


def add_history_entry(
    event_description: str,
    interests: Sequence[str],
    themes: Sequence[str],
    starter: str,
    db_path: Optional[Path] = None,
) -> int:
    with get_connection(db_path) as conn:
        cur = conn.execute(
            """
            INSERT INTO history (event_description, interests, themes, starter, useful, created_at)
            VALUES (?, ?, ?, ?, NULL, ?)
            """,
            (
                event_description,
                ",".join(interests),
                ",".join(themes),
                starter,
                datetime.now(timezone.utc).isoformat(),
            ),
        )
        conn.commit()
        return int(cur.lastrowid)


def update_feedback(history_id: int, useful: bool, db_path: Optional[Path] = None) -> bool:
    with get_connection(db_path) as conn:
        cur = conn.execute(
            "UPDATE history SET useful = ? WHERE id = ?",
            (1 if useful else 0, history_id),
        )
        conn.commit()
        return cur.rowcount > 0


def get_history(limit: int = 50, db_path: Optional[Path] = None) -> List[dict]:
    with get_connection(db_path) as conn:
        rows = conn.execute(
            "SELECT * FROM history ORDER BY id DESC LIMIT ?", (limit,)
        ).fetchall()
        return [dict(row) for row in rows]


Writing backend/database.py


In [7]:
%%writefile backend/services/__init__.py


Writing backend/services/__init__.py


In [8]:
%%writefile backend/services/theme_extractor.py
"""Extracts high-level themes from an event description using a DistilBERT
model fine-tuned for natural language inference, run in zero-shot
classification mode against a bank of candidate networking/industry topics.
"""
from typing import List, Optional

from transformers import pipeline

DEFAULT_CANDIDATE_LABELS = [
    "artificial intelligence",
    "machine learning",
    "sustainability",
    "climate change",
    "urban planning",
    "healthcare",
    "biotechnology",
    "finance",
    "blockchain",
    "cybersecurity",
    "data science",
    "entrepreneurship",
    "marketing",
    "education",
    "policy and governance",
    "design",
    "arts and culture",
    "robotics",
    "cloud computing",
    "renewable energy",
    "supply chain",
    "product management",
]


class ThemeExtractor:
    """Wraps a HuggingFace zero-shot-classification pipeline built on a
    DistilBERT NLI checkpoint. The model is loaded lazily on first use so
    importing this module (e.g. for tests) stays fast.
    """

    def __init__(self, model_name: str = "typeform/distilbert-base-uncased-mnli", device: int = -1):
        self.model_name = model_name
        self.device = device
        self._classifier = None

    @property
    def classifier(self):
        if self._classifier is None:
            self._classifier = pipeline(
                "zero-shot-classification",
                model=self.model_name,
                device=self.device,
            )
        return self._classifier

    def extract_themes(
        self,
        text: str,
        extra_labels: Optional[List[str]] = None,
        top_k: int = 3,
        score_threshold: float = 0.15,
    ) -> List[str]:
        """Return up to `top_k` theme labels for `text`.

        `extra_labels` (typically the user's stated interests) are merged
        into the candidate label pool so themes the model would otherwise
        miss can still surface if they match the description well.
        """
        if not text or not text.strip():
            return []

        labels = list(DEFAULT_CANDIDATE_LABELS)
        for label in extra_labels or []:
            label = label.strip().lower()
            if label and label not in labels:
                labels.append(label)

        result = self.classifier(text, candidate_labels=labels, multi_label=True)
        themes = [
            label
            for label, score in zip(result["labels"], result["scores"])
            if score >= score_threshold
        ][:top_k]

        # Guarantee at least one theme even if nothing cleared the threshold.
        if not themes and result["labels"]:
            themes = result["labels"][:top_k]

        return themes


Writing backend/services/theme_extractor.py


In [9]:
%%writefile backend/services/starter_generator.py
"""Generates conversation starters with GPT-2, conditioned on the themes
detected in the event description and the user's stated interests.

GPT-2 (base) is a small, un-instruction-tuned model, so raw generations can
be noisy. To keep the demo reliable this module:
  1. Prompts GPT-2 with a templated context.
  2. Samples several continuations and keeps only sentence-shaped output.
  3. Pads out to the requested count with hand-written template starters
     (personalized with the detected themes/interests) if GPT-2 doesn't
     produce enough usable candidates.
"""
import re
from typing import List, Sequence

from transformers import pipeline, set_seed

FALLBACK_TEMPLATES = [
    "What got you interested in {theme}, and how did you end up focusing on it?",
    "I noticed this event touches on {theme} - what's the most exciting problem you're tackling there right now?",
    "Given your work, how do you see {theme} evolving over the next few years?",
    "What drew you to {interest}, and how does it connect with {theme} for you?",
    "If you had to explain why {theme} matters to someone outside the field, what would you say?",
]


class StarterGenerator:
    def __init__(self, model_name: str = "gpt2", device: int = -1):
        self.model_name = model_name
        self.device = device
        self._generator = None

    @property
    def generator(self):
        if self._generator is None:
            self._generator = pipeline("text-generation", model=self.model_name, device=self.device)
            set_seed(42)
        return self._generator

    @staticmethod
    def _build_prompt(themes: Sequence[str], interests: Sequence[str]) -> str:
        theme_str = ", ".join(themes) if themes else "the event's topic"
        interest_str = ", ".join(interests) if interests else "your professional interests"
        return (
            f"At a networking event about {theme_str}, a thoughtful conversation starter "
            f"connecting to {interest_str} is: \""
        )

    @staticmethod
    def _clean_candidates(text: str) -> List[str]:
        chunks = re.split(r"(?<=[.?!])\s+", text)
        cleaned = []
        for chunk in chunks:
            chunk = chunk.strip().strip('"')
            words = chunk.split()
            if 6 <= len(words) <= 30:
                if not chunk.endswith((".", "?", "!")):
                    chunk += "?"
                cleaned.append(chunk)
        return cleaned

    def generate_starters(
        self,
        themes: Sequence[str],
        interests: Sequence[str],
        num_starters: int = 3,
    ) -> List[str]:
        themes = list(themes) or []
        interests = list(interests) or []
        prompt = self._build_prompt(themes, interests)
        starters: List[str] = []

        try:
            outputs = self.generator(
                prompt,
                max_new_tokens=40,
                num_return_sequences=num_starters,
                do_sample=True,
                top_p=0.92,
                temperature=0.9,
                pad_token_id=self.generator.tokenizer.eos_token_id,
            )
            for out in outputs:
                generated = out["generated_text"][len(prompt):]
                candidates = self._clean_candidates(generated)
                if candidates:
                    starters.append(candidates[0])
        except Exception:
            # Model unavailable/failed to load - fall back to templates below.
            starters = []

        # De-duplicate while preserving order.
        seen = set()
        unique_starters = []
        for s in starters:
            key = s.lower()
            if key not in seen:
                seen.add(key)
                unique_starters.append(s)
        starters = unique_starters

        # Pad with personalized templates if GPT-2 didn't produce enough.
        idx = 0
        while len(starters) < num_starters and idx < 20:
            theme = themes[idx % len(themes)] if themes else "this field"
            interest = interests[idx % len(interests)] if interests else "your work"
            template = FALLBACK_TEMPLATES[idx % len(FALLBACK_TEMPLATES)]
            candidate = template.format(theme=theme, interest=interest)
            if candidate.lower() not in seen:
                starters.append(candidate)
                seen.add(candidate.lower())
            idx += 1

        return starters[:num_starters]


Writing backend/services/starter_generator.py


In [10]:
%%writefile backend/services/fact_checker.py
"""Quick fact-verification / reference lookup backed by the Wikipedia API.

This does not attempt full claim verification - it retrieves a concise,
reliable summary of the topic so the user can sanity-check what they're
about to bring up in conversation, per Scenario 2 in the product spec.
"""
from typing import Dict, List, Optional

import wikipedia


class FactChecker:
    def __init__(self, sentences: int = 3, lang: str = "en"):
        self.sentences = sentences
        wikipedia.set_lang(lang)

    def check(self, query: str) -> Dict[str, object]:
        if not query or not query.strip():
            return {"query": query, "found": False, "summary": None, "url": None, "options": []}

        try:
            summary = wikipedia.summary(query, sentences=self.sentences, auto_suggest=True)
            page = wikipedia.page(query, auto_suggest=True)
            return {
                "query": query,
                "found": True,
                "summary": summary,
                "url": page.url,
                "options": [],
            }
        except wikipedia.exceptions.DisambiguationError as e:
            options: List[str] = list(e.options)[:5]
            return {"query": query, "found": False, "summary": None, "url": None, "options": options}
        except wikipedia.exceptions.PageError:
            return {"query": query, "found": False, "summary": None, "url": None, "options": []}
        except Exception as e:  # network errors, etc.
            return {
                "query": query,
                "found": False,
                "summary": f"Lookup failed: {e}",
                "url": None,
                "options": [],
            }

Writing backend/services/fact_checker.py


In [11]:
%%writefile backend/main.py
"""FastAPI backend for the Personalized Networking Assistant."""
from typing import List

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

from backend import database
from backend.models import (
    FactCheckRequest,
    FactCheckResponse,
    FeedbackRequest,
    GenerateStartersRequest,
    GenerateStartersResponse,
    HistoryItem,
    StarterItem,
)
from backend.services.fact_checker import FactChecker
from backend.services.starter_generator import StarterGenerator
from backend.services.theme_extractor import ThemeExtractor

app = FastAPI(
    title="Personalized Networking Assistant API",
    description="Generates tailored conversation starters and quick fact checks for networking events.",
    version="1.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

theme_extractor = ThemeExtractor()
starter_generator = StarterGenerator()
fact_checker = FactChecker()


@app.on_event("startup")
def on_startup() -> None:
    database.init_db()


@app.get("/health")
def health() -> dict:
    return {"status": "ok"}


@app.post("/api/generate-starters", response_model=GenerateStartersResponse)
def generate_starters(payload: GenerateStartersRequest) -> GenerateStartersResponse:
    themes = theme_extractor.extract_themes(
        payload.event_description, extra_labels=payload.interests
    )
    starters = starter_generator.generate_starters(
        themes, payload.interests, num_starters=payload.num_starters
    )
    if not starters:
        raise HTTPException(status_code=500, detail="Could not generate conversation starters.")

    items: List[StarterItem] = []
    for starter in starters:
        history_id = database.add_history_entry(
            payload.event_description, payload.interests, themes, starter
        )
        items.append(StarterItem(id=history_id, starter=starter))

    return GenerateStartersResponse(themes=themes, starters=items)


@app.post("/api/fact-check", response_model=FactCheckResponse)
def fact_check(payload: FactCheckRequest) -> FactCheckResponse:
    result = fact_checker.check(payload.query)
    return FactCheckResponse(**result)


@app.post("/api/feedback")
def feedback(payload: FeedbackRequest) -> dict:
    updated = database.update_feedback(payload.history_id, payload.useful)
    if not updated:
        raise HTTPException(status_code=404, detail="History entry not found.")
    return {"status": "updated"}


@app.get("/api/history", response_model=List[HistoryItem])
def history(limit: int = 50) -> List[HistoryItem]:
    rows = database.get_history(limit=limit)
    return [
        HistoryItem(
            id=row["id"],
            event_description=row["event_description"],
            interests=row["interests"] or "",
            themes=row["themes"] or "",
            starter=row["starter"],
            useful=(bool(row["useful"]) if row["useful"] is not None else None),
            created_at=row["created_at"],
        )
        for row in rows
    ]


Writing backend/main.py


In [12]:
%%writefile frontend/app.py
"""Streamlit frontend for the Personalized Networking Assistant.

Talks to the FastAPI backend over HTTP. Set the BACKEND_URL environment
variable if the API isn't running on localhost:8000 (e.g. when tunneling
through ngrok in Colab, the backend still stays on localhost since both
processes run in the same VM - only the Streamlit port needs a public URL).
"""
import os

import requests
import streamlit as st

API_BASE_URL = os.environ.get("BACKEND_URL", "http://localhost:8000")

st.set_page_config(page_title="Personalized Networking Assistant", page_icon="🤝", layout="centered")
st.title("🤝 Personalized Networking Assistant")
st.caption("AI-generated conversation starters, tailored to the event and to you.")

tab1, tab2, tab3 = st.tabs(["✨ Generate Starters", "🔎 Fact Check", "🕘 History"])

# ---------------------------------------------------------------- Tab 1 --
with tab1:
    st.subheader("Generate tailored conversation starters")
    event_description = st.text_area(
        "Event description", placeholder="e.g. AI for Sustainable Cities"
    )
    interests_raw = st.text_input(
        "Your interests (comma-separated)", placeholder="climate change, urban planning"
    )
    num_starters = st.slider("Number of starters", 1, 5, 3)

    if st.button("Generate", type="primary"):
        if not event_description.strip():
            st.warning("Please enter an event description.")
        else:
            interests = [i.strip() for i in interests_raw.split(",") if i.strip()]
            data = None
            with st.spinner("Extracting themes and generating starters..."):
                try:
                    resp = requests.post(
                        f"{API_BASE_URL}/api/generate-starters",
                        json={
                            "event_description": event_description,
                            "interests": interests,
                            "num_starters": num_starters,
                        },
                        timeout=120,
                    )
                    resp.raise_for_status()
                    data = resp.json()
                except Exception as e:
                    st.error(f"Request failed: {e}")

            if data:
                st.markdown("**Detected themes:** " + ", ".join(data["themes"]))
                for item in data["starters"]:
                    col1, col2, col3 = st.columns([6, 1, 1])
                    with col1:
                        st.write(f"💬 {item['starter']}")
                    with col2:
                        if st.button("👍", key=f"up_{item['id']}"):
                            requests.post(
                                f"{API_BASE_URL}/api/feedback",
                                json={"history_id": item["id"], "useful": True},
                            )
                            st.success("Thanks!")
                    with col3:
                        if st.button("👎", key=f"down_{item['id']}"):
                            requests.post(
                                f"{API_BASE_URL}/api/feedback",
                                json={"history_id": item["id"], "useful": False},
                            )
                            st.info("Got it.")

# ---------------------------------------------------------------- Tab 2 --
with tab2:
    st.subheader("Quick fact verification")
    query = st.text_input("Topic to fact-check", placeholder="blockchain in healthcare")
    if st.button("Check facts"):
        if not query.strip():
            st.warning("Please enter a topic.")
        else:
            data = None
            with st.spinner("Looking this up on Wikipedia..."):
                try:
                    resp = requests.post(
                        f"{API_BASE_URL}/api/fact-check", json={"query": query}, timeout=60
                    )
                    resp.raise_for_status()
                    data = resp.json()
                except Exception as e:
                    st.error(f"Request failed: {e}")

            if data:
                if data["found"]:
                    st.write(data["summary"])
                    st.markdown(f"[Read more on Wikipedia]({data['url']})")
                elif data["options"]:
                    st.info("Your query is ambiguous. Did you mean:")
                    for option in data["options"]:
                        st.write(f"- {option}")
                else:
                    st.warning("No reliable reference found for this topic.")

# ---------------------------------------------------------------- Tab 3 --
with tab3:
    st.subheader("Past conversation starters")
    st.button("Refresh history")
    try:
        resp = requests.get(f"{API_BASE_URL}/api/history", timeout=30)
        resp.raise_for_status()
        history = resp.json()
    except Exception as e:
        st.error(f"Could not load history: {e}")
        history = []

    if not history:
        st.caption("No history yet - generate some starters first.")

    for h in history:
        useful_icon = "👍" if h["useful"] is True else ("👎" if h["useful"] is False else "•")
        st.markdown(
            f"**{h['event_description']}** _(themes: {h['themes']})_ {useful_icon}\n\n> {h['starter']}"
        )
        st.divider()


Writing frontend/app.py


In [13]:
%%writefile tests/__init__.py


Writing tests/__init__.py


In [14]:
%%writefile tests/test_theme_extractor.py
import pytest

from backend.services.theme_extractor import ThemeExtractor


def test_extract_themes_empty_text():
    extractor = ThemeExtractor()
    assert extractor.extract_themes("") == []
    assert extractor.extract_themes("   ") == []


@pytest.mark.slow
def test_extract_themes_real_model():
    """Loads the actual DistilBERT NLI model - slow, run explicitly with:
    pytest -m slow
    """
    extractor = ThemeExtractor()
    themes = extractor.extract_themes(
        "AI for Sustainable Cities", extra_labels=["climate change", "urban planning"]
    )
    assert isinstance(themes, list)
    assert len(themes) > 0
    assert all(isinstance(t, str) for t in themes)


Writing tests/test_theme_extractor.py


In [15]:
%%writefile tests/test_starter_generator.py
from backend.services.starter_generator import StarterGenerator


def test_clean_candidates_filters_length():
    gen = StarterGenerator()
    text = (
        "Too short. "
        "This is a properly sized candidate sentence for testing purposes here. "
        "This one goes on for way too long and should be filtered out because it exceeds the thirty word ceiling that the cleaning function enforces on every single generated candidate sentence in this test case."
    )
    cleaned = gen._clean_candidates(text)
    assert len(cleaned) == 1
    assert 6 <= len(cleaned[0].split()) <= 31


def test_generate_starters_fallback_when_model_unavailable(monkeypatch):
    import backend.services.starter_generator as sg_module

    def fake_pipeline(*args, **kwargs):
        raise RuntimeError("model unavailable in this environment")

    monkeypatch.setattr(sg_module, "pipeline", fake_pipeline)

    gen = sg_module.StarterGenerator()
    starters = gen.generate_starters(["artificial intelligence"], ["climate change"], num_starters=3)

    assert len(starters) == 3
    assert all(isinstance(s, str) and s.strip() for s in starters)
    # fallback templates are personalized with the theme
    assert any("artificial intelligence" in s for s in starters)


def test_generate_starters_deduplicates(monkeypatch):
    import backend.services.starter_generator as sg_module

    class FakeTokenizer:
        eos_token_id = 0

    class FakeGenerator:
        tokenizer = FakeTokenizer()

        def __call__(self, prompt, **kwargs):
            n = kwargs.get("num_return_sequences", 1)
            # every generation is identical -> should be de-duplicated
            return [
                {"generated_text": prompt + "What excites you most about this topic?"}
                for _ in range(n)
            ]

    monkeypatch.setattr(sg_module, "pipeline", lambda *a, **k: FakeGenerator())
    monkeypatch.setattr(sg_module, "set_seed", lambda *a, **k: None)

    gen = sg_module.StarterGenerator()
    starters = gen.generate_starters(["AI"], ["data"], num_starters=3)

    assert len(starters) == 3
    # only one unique GPT-2 style starter should appear once, rest padded by fallback
    assert len([s for s in starters if "excites you most" in s]) == 1


Writing tests/test_starter_generator.py


In [16]:
%%writefile tests/test_fact_checker.py
from backend.services.fact_checker import FactChecker


def test_fact_check_empty_query():
    checker = FactChecker()
    result = checker.check("")
    assert result["found"] is False
    assert result["summary"] is None


def test_fact_check_success_mocked(monkeypatch):
    import backend.services.fact_checker as fc_module

    class FakePage:
        url = "https://en.wikipedia.org/wiki/Blockchain"

    monkeypatch.setattr(
        fc_module.wikipedia,
        "summary",
        lambda query, sentences, auto_suggest: "Blockchain is a distributed ledger technology.",
    )
    monkeypatch.setattr(fc_module.wikipedia, "page", lambda query, auto_suggest: FakePage())

    checker = FactChecker()
    result = checker.check("blockchain")

    assert result["found"] is True
    assert "Blockchain" in result["summary"]
    assert result["url"].startswith("https://en.wikipedia.org")


def test_fact_check_disambiguation_mocked(monkeypatch):
    import backend.services.fact_checker as fc_module

    def raise_disambiguation(query, sentences, auto_suggest):
        raise fc_module.wikipedia.exceptions.DisambiguationError("mercury", ["Mercury (planet)", "Mercury (element)"])

    monkeypatch.setattr(fc_module.wikipedia, "summary", raise_disambiguation)

    checker = FactChecker()
    result = checker.check("mercury")

    assert result["found"] is False
    assert len(result["options"]) > 0


Writing tests/test_fact_checker.py


In [17]:
%%writefile tests/test_api.py
from fastapi.testclient import TestClient

import backend.main as main_module

client = TestClient(main_module.app)


def test_health():
    resp = client.get("/health")
    assert resp.status_code == 200
    assert resp.json() == {"status": "ok"}


def test_generate_starters(monkeypatch):
    monkeypatch.setattr(
        main_module.theme_extractor,
        "extract_themes",
        lambda text, extra_labels=None, top_k=3, score_threshold=0.15: ["artificial intelligence", "sustainability"],
    )
    monkeypatch.setattr(
        main_module.starter_generator,
        "generate_starters",
        lambda themes, interests, num_starters=3: ["Starter one?", "Starter two?", "Starter three?"][:num_starters],
    )
    monkeypatch.setattr(main_module.database, "add_history_entry", lambda *a, **k: 1)

    resp = client.post(
        "/api/generate-starters",
        json={
            "event_description": "AI for Sustainable Cities",
            "interests": ["climate change"],
            "num_starters": 2,
        },
    )

    assert resp.status_code == 200
    data = resp.json()
    assert data["themes"] == ["artificial intelligence", "sustainability"]
    assert len(data["starters"]) == 2


def test_generate_starters_validation_error():
    resp = client.post("/api/generate-starters", json={"event_description": "ok"})
    # event_description passes min_length=3, so this actually succeeds through
    # to generation; a genuinely invalid payload (missing field) is used below.
    resp2 = client.post("/api/generate-starters", json={"interests": ["ai"]})
    assert resp2.status_code == 422


def test_fact_check(monkeypatch):
    monkeypatch.setattr(
        main_module.fact_checker,
        "check",
        lambda query: {
            "query": query,
            "found": True,
            "summary": "Some reliable summary.",
            "url": "https://en.wikipedia.org/wiki/Test",
            "options": [],
        },
    )
    resp = client.post("/api/fact-check", json={"query": "test topic"})
    assert resp.status_code == 200
    assert resp.json()["found"] is True


def test_feedback_not_found(monkeypatch):
    monkeypatch.setattr(main_module.database, "update_feedback", lambda *a, **k: False)
    resp = client.post("/api/feedback", json={"history_id": 999, "useful": True})
    assert resp.status_code == 404


def test_history(monkeypatch):
    monkeypatch.setattr(
        main_module.database,
        "get_history",
        lambda limit=50: [
            {
                "id": 1,
                "event_description": "AI Summit",
                "interests": "ai",
                "themes": "artificial intelligence",
                "starter": "What excites you about AI?",
                "useful": 1,
                "created_at": "2026-01-01T00:00:00+00:00",
            }
        ],
    )
    resp = client.get("/api/history")
    assert resp.status_code == 200
    data = resp.json()
    assert len(data) == 1
    assert data[0]["useful"] is True


Writing tests/test_api.py


In [18]:
%%writefile pytest.ini
[pytest]
pythonpath = .
markers =
    slow: marks tests that load real ML models (deselect with -m "not slow")


Writing pytest.ini


In [19]:
%%writefile requirements.txt
fastapi==0.115.0
uvicorn[standard]==0.30.6
streamlit==1.38.0
transformers==4.44.2
torch==2.4.0 torchvision==0.19.0
wikipedia==1.4.0
pydantic==2.9.2
pytest==8.3.2
httpx==0.27.2
requests==2.32.3
pyngrok==7.2.0
nest-asyncio==1.6.0


Writing requirements.txt


## 4. Run the test suite

The fast tests (default) mock the ML models so they run in seconds. Add `-m slow`
to also exercise the real DistilBERT model.

In [20]:
!cd /content/personalized-networking-assistant && python -m pytest -q


..............                                                           [100%]
=============================== warnings summary ===============================
tests/test_theme_extractor.py::test_extract_themes_real_model
  /usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
14 passed, 1 warning in 37.35s


## 5. Start the FastAPI backend in a background thread

In [21]:
import threading, time, requests
import nest_asyncio
import uvicorn

nest_asyncio.apply()

import sys
sys.path.insert(0, "/content/personalized-networking-assistant")

from backend.main import app as fastapi_app

def run_backend():
    uvicorn.run(fastapi_app, host="0.0.0.0", port=8000, log_level="warning")

backend_thread = threading.Thread(target=run_backend, daemon=True)
backend_thread.start()

# wait for the server to come up
for _ in range(30):
    try:
        r = requests.get("http://localhost:8000/health", timeout=2)
        if r.status_code == 200:
            print("Backend is up:", r.json())
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print("Backend did not respond in time - check the logs above.")


Exception in thread Thread-3 (run_backend):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_2382/225440202.py", line 13, in run_backend
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/main.py", line 577, in run
    server.run()
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 65, in run
    return asyncio.run(self.serve(sockets=sockets))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/nest_asyncio.py", line 26, in run
    loop = asyncio.get_event_loop()
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/nest_asyncio.py", line 40, in _get_event_loop
    loop = events.get_event_loop_policy().get_event_loop()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fi

Backend did not respond in time - check the logs above.


In [22]:
from pyngrok import ngrok, conf

NGROK_AUTHTOKEN = "3HomPSnyLpEUPYAobPNDmJxc7XY_43T2BM327GZAZrdjhXgud"  # <-- paste your token here

if NGROK_AUTHTOKEN:
    conf.get_default().auth_token = NGROK_AUTHTOKEN
else:
    print("No authtoken set - ngrok may rate-limit or refuse the tunnel. "
          "Set NGROK_AUTHTOKEN above and re-run this cell if it fails.")


In [23]:
import subprocess, time, os

env = os.environ.copy()
env["BACKEND_URL"] = "http://localhost:8000"

streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", "/content/personalized-networking-assistant/frontend/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
    ],
    env=env,
)
time.sleep(8)  # give streamlit a moment to boot

# close any previous tunnels so re-running this cell doesn't stack them
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

public_url = ngrok.connect(8501, "http")
print("🎉 Streamlit app is live at:", public_url)
print("Open this URL in your browser to use and record the demo.")


🎉 Streamlit app is live at: NgrokTunnel: "https://appointee-default-engaged.ngrok-free.dev" -> "http://localhost:8501"
Open this URL in your browser to use and record the demo.
